In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os

df_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(df_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)
# huge missing so fill with mean

for col in missing_data["Column"]:
  df[col].fillna(df[col].mean())

In [ ]:
# Task 2: Write your code here:
duplicates = df.duplicated().sum()

if duplicates > 0:
  df.drop_duplicates(inplace=True)

# for check:
df.info()

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder #import OneHotEncoder


categorical_cols = df.select_dtypes(include=["object"]).columns

label_encoder = LabelEncoder()

for col in categorical_cols:
  df[col] = label_encoder.fit_transform(df[col]) # Encode

# for check:
df.info()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[cols] = scaler.fit_transform(df[cols])
df.head()


In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns

def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

# imbalance

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1)
y = df["Target"]

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

model = CatBoostClassifier(verbose=0, n_estimators=320)

lr_accuracy = []
lr_f1 = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  lr_accuracy.append(accuracy_score(y_test, y_pred))
  lr_f1.append(f1_score(y_test, y_pred))

print(f"Accuracy:  {np.mean(lr_accuracy):.4f}")
print(f"F1 Score:  {np.mean(lr_f1):.4f}")

In [ ]:
# Task 1: Write your code here:
feature_cols = []
for col in X.columns:
  feature_cols.append(col)

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
feature_importance.head(1)
print(f"The Golden Feture is P_2")

In [ ]:
# Task Bonus: Write your code here:
X = df["P_2"]
y = df["Target"]


n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

model = CatBoostClassifier(verbose=0, n_estimators=320)

lr_accuracy = []
lr_f1 = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  lr_accuracy.append(accuracy_score(y_test, y_pred))
  lr_f1.append(f1_score(y_test, y_pred))

print(f"Accuracy:  {np.mean(lr_accuracy):.4f}")
print(f"F1 Score:  {np.mean(lr_f1):.4f}")
